# Gold Layer - Ops FDA Performance Report

**Stakeholder:** Operations<br>
**Question:** What is our first-attempt delivery success (FDA) rate, and where is it weak?

In [1]:
import pandas as pd

final_df = pd.read_csv('sample_data/output/final_shipments.csv')
final_df['booking_timestamp'] = pd.to_datetime(final_df['booking_timestamp'])
final_df['is_breached'] = final_df['is_breached'].astype('boolean')

final_df.head()

,shipment_id,booking_timestamp,promised_delivery_date,region,customer_type,service_tier,is_guaranteed,weight_kg,fda,first_attempt_date,actual_delivery_date,invoice_date,base_rate,fuel_surcharge_pct,fuel_surcharge_amount,guarantee_fee,total_billed,is_breached,credit_liability
0,SHP-00000000,2011-07-13 08:40:56.139636,2011-07-16,Atlantic,RESIDENTIAL,GROUND,False,3.62,1.0,2011-07-14,2011-07-14,2011-07-14,8.00,0.1478,1.18,0.0,9.18,False,0.0
1,SHP-00000001,2011-07-13 17:52:54.265014,2011-07-16,Calgary-Edmonton,RESIDENTIAL,GROUND,False,23.23,1.0,2011-07-14,2011-07-14,2011-07-14,27.88,0.1478,4.12,0.0,32.00,False,0.0
2,SHP-00000002,2011-07-13 16:58:01.421036,2011-07-16,GTA,COMMERCIAL,GROUND,False,18.53,1.0,2011-07-14,2011-07-14,2011-07-14,22.24,0.1478,3.29,0.0,25.53,False,0.0
3,SHP-00000003,2011-07-13 07:07:17.617848,2011-07-16,GTA,RESIDENTIAL,GROUND,False,24.06,0.0,2011-07-14,2011-07-15,2011-07-13,28.87,0.1478,4.27,0.0,33.14,False,0.0
4,SHP-00000004,2011-07-13 09:43:06.971432,2011-07-13,Montreal,RESIDENTIAL,SAME_DAY,False,1.71,0.0,2011-07-13,2011-07-15,2011-07-13,25.00,0.1478,3.69,0.0,28.69,True,0.0


In [2]:
# day_of_week keyed off the attempt date=
final_df['first_attempt_date'] = pd.to_datetime(final_df['first_attempt_date'])
final_df['quarter'] = final_df['booking_timestamp'].dt.to_period('Q').astype(str)
final_df['day_of_week'] = final_df['first_attempt_date'].dt.day_name()

### FDA Rate by Quarter, Region, Tier, and Customer Type

In [3]:
# broad segment cut of the FDA rate and breach rate
# `is_breached` is filled with `False` before averaging since shipments still in transit (null breach status) haven't breached yet.
gold_ops_fda_report = final_df.groupby(['quarter', 'region', 'service_tier', 'customer_type']).agg(
    total_shipments=('shipment_id', 'count'),
    avg_fda_rate=('fda', 'mean'),
    breach_rate=('is_breached', lambda x: x.fillna(False).mean()),
).reset_index()

gold_ops_fda_report

,quarter,region,service_tier,customer_type,total_shipments,avg_fda_rate,breach_rate
0,2011Q3,Atlantic,EXPRESS,COMMERCIAL,74,0.905405,0.067568
1,2011Q3,Atlantic,EXPRESS,RESIDENTIAL,131,0.778626,0.183206
2,2011Q3,Atlantic,GROUND,COMMERCIAL,211,0.928571,0.023697
3,2011Q3,Atlantic,GROUND,RESIDENTIAL,438,0.766055,0.034247
4,2011Q3,Atlantic,SAME_DAY,COMMERCIAL,28,0.857143,0.142857
...,...,...,...,...,...,...,...
1825,2026Q3,Vancouver,EXPRESS,RESIDENTIAL,80,0.721519,0.1875
1826,2026Q3,Vancouver,GROUND,COMMERCIAL,151,0.854305,0.02649
1827,2026Q3,Vancouver,GROUND,RESIDENTIAL,249,0.694779,0.032129
1828,2026Q3,Vancouver,SAME_DAY,COMMERCIAL,23,0.909091,0.086957


### Overall FDA Rate

In [4]:
# headline FDA metric for the report - moved here from Silver since it's a business KPI, not a data-quality check
rate = final_df['fda'].mean() * 100
print(f"First Delivery Attempt Success Rate: {rate:.2f}%")

First Delivery Attempt Success Rate: 80.82%


### FDA Rate by Day of Week and Customer Type

In [5]:
# narrower cut isolating the day-of-week effect within each customer type
gold_ops_fda_by_dow = final_df.groupby(['day_of_week', 'customer_type']).agg(
    total_shipments=('shipment_id', 'count'),
    avg_fda_rate=('fda', 'mean'),
).reset_index()

gold_ops_fda_by_dow

,day_of_week,customer_type,total_shipments,avg_fda_rate
0,Friday,COMMERCIAL,58425,0.896277
1,Friday,RESIDENTIAL,108194,0.730946
2,Monday,COMMERCIAL,35689,0.901734
3,Monday,RESIDENTIAL,66760,0.721001
4,Saturday,COMMERCIAL,58567,0.900951
5,Saturday,RESIDENTIAL,108199,0.759878
6,Sunday,COMMERCIAL,56288,0.899215
7,Sunday,RESIDENTIAL,104104,0.750192
8,Thursday,COMMERCIAL,58225,0.898635
9,Thursday,RESIDENTIAL,108019,0.779937


In [6]:
gold_ops_fda_by_dow[gold_ops_fda_by_dow['customer_type'] == 'RESIDENTIAL'].sort_values('avg_fda_rate')

,day_of_week,customer_type,total_shipments,avg_fda_rate
3,Monday,RESIDENTIAL,66760,0.721001
1,Friday,RESIDENTIAL,108194,0.730946
7,Sunday,RESIDENTIAL,104104,0.750192
5,Saturday,RESIDENTIAL,108199,0.759878
9,Thursday,RESIDENTIAL,108019,0.779937
11,Tuesday,RESIDENTIAL,87630,0.780041
13,Wednesday,RESIDENTIAL,108302,0.780540


### Results

**RESIDENTIAL** shows a clear day-of-week pattern, with the two weakest days being:
- **Monday: 72.1%** avg FDA rate (lowest)
- **Friday: 73.1%** avg FDA rate (second lowest)

vs. a Tue–Wed–Thu baseline of ~78–78.5%.<br><br>

**COMMERCIAL** stays essentially flat across all seven days (~89.6–90.2%), so the day-of-week effect is specific to residential deliveries - consistent with residential FDA depending on someone being home, which is least likely on Monday and Friday.